# 8 — Lipid lateral diffusion coefficient

**A trajectory is required.** Diffusion is measured from how far lipids move
over **time** (mean-squared displacement), so a single `.gro` frame cannot
give it. Once you have a **no-jump** trajectory
(`gmx trjconv -pbc nojump`), set `TRAJ` below and run.

Method: per-lipid COM, per-leaflet drift removed, lateral MSD fitted as
`MSD_xy = 4 D t`.

## Setup

In [ ]:
# !pip install "MDAnalysis>=2.4" numpy matplotlib
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt
import MDAnalysis as mda

## Config

In [ ]:
TOPOLOGY = "md.tpr"
TRAJ     = "md_nojump.xtc"   # REQUIRED: a pbc-nojump trajectory
LIPIDS   = ["DOPC","DOPA"]
HEAD     = "P"; STEP = 1
FITMIN, FITMAX = 0.2, 0.6     # fit window as fractions of the lag range

## Compute MSD → D

In [ ]:
import os
assert os.path.exists(TRAJ), "No trajectory found -- diffusion needs one (see note above)."
ud=mda.Universe(TOPOLOGY,TRAJ)
present=[r for r in LIPIDS if len(ud.select_atoms(f"resname {r}"))>0]
lip=ud.select_atoms("resname "+" ".join(present)); residues=lip.residues
resnames=np.array([r.resname for r in residues])
Pd=ud.select_atoms(f"name {HEAD} and resname "+" ".join(present))
ud.trajectory[0]; zc0=Pd.positions[:,2].mean()
pz={a.resid:a.position[2] for a in Pd}
leaf=np.array([0 if pz.get(r.resid,zc0)>=zc0 else 1 for r in residues])
times,coms=[],[]
for ts in ud.trajectory[::STEP]:
    times.append(ts.time)
    com=np.array([r.atoms.center_of_mass()[:2] for r in residues])
    for lf in (0,1):
        m=leaf==lf
        if m.any(): com[m]-=com[m].mean(axis=0)
    coms.append(com)
times=np.array(times); coms=np.array(coms); T=len(times)
dt=np.mean(np.diff(times)) if T>1 else 1.0
plt.figure(figsize=(7,4.5)); print(f"{'lipid':<8}{'D (cm^2/s)':>14}{'D (1e-7)':>12}")
for r in present:
    idx=np.where(resnames==r)[0]
    if idx.size==0: continue
    sub=coms[:,idx,:]; msd=np.zeros(T)
    for lag in range(1,T): msd[lag]=((sub[lag:]-sub[:-lag])**2).sum(axis=2).mean()
    lags=np.arange(T)*dt; np.savetxt(f"msd_{r}.dat",np.column_stack([lags,msd]),header="lag_ps msd_A2")
    i0,i1=max(1,int(FITMIN*T)),int(FITMAX*T)
    D=np.polyfit(lags[i0:i1],msd[i0:i1],1)[0]/4*1e-4
    print(f"{r:<8}{D:14.3e}{D*1e7:12.3f}"); plt.plot(lags,msd,label=r)
plt.xlabel("lag (ps)"); plt.ylabel(r"MSD$_{xy}$ ($\AA^2$)"); plt.legend()
plt.grid(alpha=.3); plt.title("Lateral MSD"); plt.show()